# ```Imports```

In [10]:
import torch
# Torch will handle all the established maths behind the curtain fro us, lot of calculus and tensors.
# let's set up owr devices now;
device='cuda' if torch.cuda.is_available() else 'cpu'

# hyper parameters
block_size=8
batch_size=4
# The avobe parameters are very very important for training.
# We will see how important they are when we scale our model and the parameters will keep getting more and more complex.


# SideEffects
print(device)
# cuda is available
#!nvidia-smi

cuda


# ```Dataset```
Let's download a small dataset from ```Project Guttenberg```(https://www.gutenberg.org)
We are going to train a ```transformer``` or a ```bigram language model``` with our preparred dataset

In [11]:
# We are opening the text in read mode with encoding 'utf-8'
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text=f.read()
# Plain text is a list of strings or char s
print(len(text))
chars=sorted(set(text))
# Sorted is not for sorting but for creating a list from a set
vocabulary_size=len(chars)
print(vocabulary_size)
print(chars)

232308
80
['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [12]:
# Now we are are going to use tokenizers with generative for loops and custom functions;
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[string_to_int[c] for c in s]
#decode=lambda l:''.join([chars[int(*j)] for j in l])
decode=lambda l:''.join([int_to_string[j] for j in l])
# All the errors because I had an extra pair of []s


In [13]:
# same as the above lambda
def encode(s):
    encode_list=[]
    for c in s:
        encode_list.append(string_to_int[c])
    return encode_list
def decode(l):
    decode_list=[]
    for j in l:
        decode_list.append(int_to_string[j])
    decoded_text=''.join(decode_list)
    return decoded_text

In [14]:
# Let's test the lambdas
encoded_hello=encode('hello')
# So now we have a conversion from characters to integers so let's decode it back into our feature
decoded_hello=decode(encoded_hello)
decoded_hello

'hello'

# ```Tokenizers```:
Right now we have been using ```character level``` tokenizer, it takes each ```character``` and converts them into an ```integer```.
We have a ```small vocabulary``` but a ```large number of tokens```(232308 characters) that we have converted into ints.
If we had a ```word level tokenizer``` meaning all the possible ```meaningful arrangements``` of group of characters.
So the ```biggest real world word``` having ```r number of characters``` will belong to ```{{nPr+np(r-1)+.....+nP(r-r)}and{all the words in the dictionary}}``` while n=80 according to above len('tf-8' encoding)
So Ton of more tokens, maybe millions or billions of tokens
Ususally we will be ```between``` character level or word level tokenizer
# ```Efficiency``` in Dataset
In the context of language model it is very important that we are efficient with our data.
Just having a giant stream might not work the best

In [15]:
# let's unleash the torch! khuuuuuuu
# Our guy wants to use torch.long so int64; he thinks it's because we will be dealing with a super long sequence of tensors, so bigger chunks in ram assignment the better
data=torch.tensor(encode(text),dtype=torch.long)
data.ndim,data[:100]
# It has only one dimensions as bunch of integers inside one set of []s

(1,
 tensor([ 1,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
         47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
          0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
          0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
         33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
         25, 38, 28,  1, 39, 30,  1, 39, 50,  9]))

# Let's talk about training and validation ```splits```
We need to ```train``` on a part of the ```corpus``` and keep remaining parts for ```validation``` or testing
If we train on the ```whole corpus```, it will ```memorize``` everything; that spoils the intelligence.
That testing part is what makes the ```adaptive nodes``` inside the black box

In [16]:
# Let's get the split at around 80% of the data and n gives us exact iteror index at the exact slice of 0.8*232308 from the start
n=int(0.8*len(data))
train_data=data[:n] # 80% of the data
val_data=data[n:] # remaing 20% of the data

# ```Bigram Bi Prefix Means 2```
like the ```in between``` aproach in animation, but different in a sense that if it has a character h then an e after that will have ```bigger probabilty``` than l given the  real world usecase of ```meaningful word making```.
As ```two letters are involved``` and thus ```weight```(of the second letter) is determined based on the ```position of two letters one after another``` it's called a bigram model.
Similar models we can generate for our ```steps lesson from mewstro``` where the second note(```target```) is weighted by the context of the first note(```context```).

# ```Block Size```
Instead of taking the whole corpus size we are going to take ```small snippets```.
We are going to ```make some predictions``` and we are going to ```decide some targets``` out of that.
Let's take a ```block size``` of 5(probably a trade off between the compression and the accuracy of the predictions) and observe two ```consecutive interlocking blocks``` in a row.

1. [...[5, 67, 21, 58, 40],35 ...]
2. [... 5,[67, 21, 58, 40, 35]...]

So the next block is ```slided over``` the other at the ```interlocked zone``` of a text_list==[67, 21, 58, 40]

In [17]:
# To implement the blocks pythonically we will start the snippet from the slice where the first_interlocking_snippet==encoded_text[:block_size] and second_interlocking_snippet==encoded_text[1:block_size+1] 
#block_size=8
x=train_data[:block_size] # x is a list
y=train_data[1:block_size+1] # y is a list
# Both x,y are the interlocking slices from the training data which starts from data[0]
for t in range(block_size): # iteration number = block_size && t=0, t+=1 on each iteration upto t=block_size-1
    # Context is the elements inside the list;
    context= x[:t+1] # assignment of context, meaning the first of the two interlocking (list|chunk|block)-populating like the sheeps from x[0:1] to x[0:block_size]
    # target is the element outside the list but the index=0 of the second (list|chunk|block)
    target=y[t]
    # We are not appending so all the assignments are being reassignments withe modified indexes on each iteration
    print(f'context=={context} gives target=={target}')

context==tensor([1]) gives target==1
context==tensor([1, 1]) gives target==28
context==tensor([ 1,  1, 28]) gives target==39
context==tensor([ 1,  1, 28, 39]) gives target==42
context==tensor([ 1,  1, 28, 39, 42]) gives target==39
context==tensor([ 1,  1, 28, 39, 42, 39]) gives target==44
context==tensor([ 1,  1, 28, 39, 42, 39, 44]) gives target==32
context==tensor([ 1,  1, 28, 39, 42, 39, 44, 32]) gives target==49


# This Isn't Scalable Yet!
This is ```sequencial```; sequencial is another way to describe ```what a cpu does as opposed to gpus```.
CPUs can do a lot of complex tasks very quickly but it only happens sequencially.
On the other hands, GPUs we can do a lot of simplersmaller computations very very quickly and in parallel.
The processors of gpus are much inferierior as units but they come in large numbers.
So what we can do is we can take each of these little blocks and then we can stack them to push these to GPUs to scale our training, a lot.

# Batches as a Hyper Parameter
For illustration let's take 4 blocks and stack them on top of one another;
```
[... ...]
[... ...]
[... ...]
[... ...]
```
We can represent this new block of 4 blocks as a new hyper parameter called batch size.
It tells us how many of these sequences or block of loops we can process in parallel. (So a 2d tensor?)


```
batch_size=[[... ...]
            [... ...]
            [... ...]
            [... ...]]
```                   

So we just went over how we get batches or rather how we can utilize this second layer of representation to accelerate the training process using the parallel processing muscle of the GPUs.
Let's take our block_size variable up at the import cell where other global variables like batch_size will be defined.
